In [8]:
import os
import glob
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import AutoVideoProcessor, AutoModel
from PIL import Image
from tqdm import tqdm

# MPS - Mac GPU Acceleration
assert torch.backends.mps.is_available(), "Pytorch MPS backend is not available. Please ensure you're running on a compatible Mac with M4 Max and have the latest PyTorch version installed."
device = torch.device("mps")
dtype = torch.float32 # MPS works best with float32 precision

print(f"🚀 M4 Max GPU Activated | Device: {device}")

🚀 M4 Max GPU Activated | Device: mps


In [2]:
class RugbyVideoDataset(Dataset):
    def __init__(self, root_dir, processor, num_frames=16):
        self.root_dir = root_dir
        self.processor = processor
        self.num_frames = num_frames
        
        # 8 labels for rugby video classification
        self.labels = ["Tackle", "Ruck", "Maul", "General_Play", "Out_of_Play", "Scrum", "Lineout", "Goal_Kick"]
        self.label2id = {label: i for i, label in enumerate(self.labels)}
        self.id2label = {i: label for label, i in self.label2id.items()}
        
        self.video_paths = []
        self.video_labels = []
        
        # Scan through each label folder and collect video paths and corresponding labels
        for label in self.labels:
            folder_path = os.path.join(root_dir, label)
            mp4_files = glob.glob(os.path.join(folder_path, "*.mp4"))
            for f in mp4_files:
                self.video_paths.append(f)
                self.video_labels.append(self.label2id[label])
                
        print(f"✅ Đã load thành công {len(self.video_paths)} video hợp lệ từ 8 nhãn.")

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        label = self.video_labels[idx]
        
        # Read video using OpenCV
        cap = cv2.VideoCapture(video_path)
        pil_frames = []
        
        # Calculate frame sampling step to get num_frames evenly spaced frames
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        step = max(1, total_frames // self.num_frames)
        
        for i in range(self.num_frames):
            cap.set(cv2.CAP_PROP_POS_FRAMES, min(i * step, total_frames - 1))
            ret, frame = cap.read()
            if not ret: break
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_frames.append(Image.fromarray(rgb_frame))
        cap.release()
        
        # If video has fewer frames than num_frames, pad with the last frame or a blank image
        while len(pil_frames) < self.num_frames:
            pil_frames.append(pil_frames[-1] if pil_frames else Image.new('RGB', (224, 224)))

        # Pass through V-JEPA Processor
        inputs = self.processor(pil_frames, return_tensors="pt")
        pixel_values = inputs['pixel_values_videos'].squeeze(0) # Xóa batch dimension dư thừa
        
        return pixel_values, torch.tensor(label, dtype=torch.long)

In [ ]:
class RugbyLinearProbe(nn.Module):
    def __init__(self, model_name="facebook/vjepa2-vitl-fpc64-256", num_classes=8):
        super().__init__()
        print(f"⏳ Khởi tạo V-JEPA 2: {model_name}...")
        self.backbone = AutoModel.from_pretrained(model_name)
        
        # Freeze backbone parameters
        for param in self.backbone.parameters():
            param.requires_grad = False
            
        embed_dim = self.backbone.config.hidden_size if hasattr(self.backbone.config, 'hidden_size') else 1024
        
        # Fine tuning head: LayerNorm + Dropout + Linear
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(0.5), # Chống Overfitting
            nn.Linear(embed_dim, num_classes)
        )

    def forward(self, pixel_values_videos):
        # Since backbone is frozen, we can disable gradient calculation for efficiency
        with torch.no_grad():
            outputs = self.backbone(pixel_values_videos=pixel_values_videos)
            video_features = outputs.last_hidden_state.mean(dim=1) # Average Pooling
            
        # Push feature through the trainable head
        logits = self.head(video_features)
        return logits

# Initialize processor and model
processor = AutoVideoProcessor.from_pretrained("facebook/vjepa2-vitl-fpc64-256")
model = RugbyLinearProbe(num_classes=8).to(device, dtype=dtype)
print("✅ Model is ready with frozen backbone and trainable head.")

video_preprocessor_config.json: 0.00B [00:00, ?B/s]

⏳ Khởi tạo V-JEPA 2: facebook/vjepa2-vitl-fpc64-256...


config.json:   0%|          | 0.00/785 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.30G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

✅ Mô hình đã sẵn sàng cho Linear Probing!


In [ ]:
DATA_DIR = "mydata" 
BATCH_SIZE = 32
NUM_FRAMES = 64
NUM_WORKERS = 0

full_dataset = RugbyVideoDataset(root_dir=DATA_DIR, processor=processor, num_frames=NUM_FRAMES)

# 80% Train, 20% Val split
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f"✅ Training: {train_size} video | Validation: {val_size} video")

✅ Đã load thành công 284 video hợp lệ từ 8 nhãn.
✅ Training: 227 video | Validation: 57 video


In [ ]:
EPOCHS = 10
LEARNING_RATE = 1e-3 # Linear probing needs a higher learning rate than full fine-tuning

criterion = nn.CrossEntropyLoss()
# Only pass the head parameters to the optimizer for updating
optimizer = torch.optim.AdamW(model.head.parameters(), lr=LEARNING_RATE)

print("🚀 Starting Linear Probing Training...")

for epoch in range(EPOCHS):
    model.train()
    total_loss, correct, total = 0, 0, 0
    
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    for pixel_values, labels in loop:
        pixel_values = pixel_values.to(device, dtype=dtype)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(pixel_values)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        loop.set_postfix(loss=loss.item(), acc=100.*correct/total)
        
    # Validation
    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for pixel_values, labels in val_loader:
            pixel_values = pixel_values.to(device, dtype=dtype)
            labels = labels.to(device)
            outputs = model(pixel_values)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
            
    val_acc = 100. * val_correct / val_total
    print(f"📊 Results Epoch {epoch+1}: Train Loss: {total_loss/len(train_loader):.4f} | Val Accuracy: {val_acc:.2f}%\n")

# Save only the linear head's state_dict since the backbone is frozen and can be reloaded from pretrained
torch.save(model.head.state_dict(), "rugby_linear_head.pth")
print("🎉 Saved linear head weights to 'rugby_linear_head.pth'")